## 1 · Synthetic weather generator

All functions take an **RGB float image in [0,1]** and return the same. `depth01` is `0 = near, 1 = far`
(so fog/dust/snow thicken with distance). Labels never change, so the synthetic images are labelled for free.

**Bugs fixed vs. the original pseudocode**
- `night`: clip to [0,1] **before** `np.power` (negative bases → NaN); RGB channel order fixed; noise added then clipped.
- `rain`: guard against a zero‑sum motion kernel; clip streak layer.
- `fog`: explicit near/far depth convention + configurable depth scale so severity is meaningful.
- All functions accept uint8 **or** float input.


In [ ]:

def _as_float_rgb(img):
    """Accept uint8/float, HxWx3, return RGB float [0,1]."""
    img = np.asarray(img).astype(np.float32)
    if img.max() > 1.5:
        img = img / 255.0
    return np.clip(img, 0.0, 1.0)

def add_fog(clean, depth01, beta=0.5, atmos=0.85, max_depth=6.0):
    clean = _as_float_rgb(clean)
    d = np.clip(depth01, 0, 1).astype(np.float32) * max_depth
    t = np.clip(np.exp(-beta * d), 0.05, 1.0)[..., None]
    return np.clip(clean * t + np.float32(atmos) * (1 - t), 0, 1)

def add_rain(clean, intensity=0.5, angle=75, streak_len=25, seed=0):
    clean = _as_float_rgb(clean); H, W = clean.shape[:2]
    rng = np.random.default_rng(seed)
    layer = np.zeros((H, W), np.float32)
    n = int(intensity * 12000)
    ys = rng.integers(0, H, n); xs = rng.integers(0, W, n)
    layer[ys, xs] = rng.uniform(0.5, 1.0, n).astype(np.float32)
    k = max(3, int(streak_len))
    kernel = np.zeros((k, k), np.float32)
    cv2.line(kernel, (k // 2, 0), (k // 2, k - 1), 1.0, 1)
    M = cv2.getRotationMatrix2D((k / 2, k / 2), angle - 90, 1.0)
    kernel = cv2.warpAffine(kernel, M, (k, k))
    s = kernel.sum(); kernel = kernel / s if s > 1e-6 else kernel
    streaks = np.clip(cv2.filter2D(layer, -1, kernel), 0, 1)
    out = clean * (1.0 - intensity * 0.2) + np.stack([streaks * 0.8] * 3, -1)
    return np.clip(out, 0, 1)

def add_night(clean, severity=0.5, seed=0):
    clean = _as_float_rgb(clean); rng = np.random.default_rng(seed)
    gamma = 1.0 + severity * 3.0
    out = np.power(np.clip(clean, 1e-6, 1.0), gamma)       # clip BEFORE power (fix)
    out[..., 0] *= (1 - severity * 0.30)                   # R down
    out[..., 1] *= (1 - severity * 0.15)                   # G down
    out[..., 2] *= (1 + severity * 0.10)                   # B up (cool cast)
    out = out + rng.normal(0, severity * 0.06, clean.shape).astype(np.float32)
    bright = (clean.mean(-1) > 0.8).astype(np.float32)
    bloom = cv2.GaussianBlur(bright, (0, 0), 12)
    out = out + bloom[..., None] * 0.25 * severity
    return np.clip(out, 0, 1)

def add_glare(clean, severity=0.5, light_xy=None):
    clean = _as_float_rgb(clean); H, W = clean.shape[:2]
    if light_xy is None:
        g = clean.mean(-1); y, x = np.unravel_index(g.argmax(), g.shape); light_xy = (int(x), int(y))
    Y, X = np.mgrid[0:H, 0:W]
    dist = np.sqrt((X - light_xy[0]) ** 2 + (Y - light_xy[1]) ** 2)
    glow = np.exp(-dist / (80 + severity * 220))
    bloom = glow * severity * 0.6
    bloom3 = np.stack([bloom, bloom * 0.95, bloom * 0.85], -1)   # warm
    return np.clip(clean + bloom3 + severity * 0.15, 0, 1)

def add_dust(clean, severity=0.5, depth01=None, seed=0, max_depth=6.0):
    """Sand / dust storm: warm ochre depth-aware veil + low-freq turbulence + desaturation."""
    clean = _as_float_rgb(clean); H, W = clean.shape[:2]
    rng = np.random.default_rng(seed)
    if depth01 is None:
        depth01 = np.tile(np.linspace(0.2, 1.0, H)[:, None], (1, W)).astype(np.float32)
    d = np.clip(depth01, 0, 1) * max_depth
    beta = 0.35 + severity * 0.9
    t = np.clip(np.exp(-beta * d), 0.05, 1.0)[..., None]
    turb = rng.random((H // 16 + 1, W // 16 + 1)).astype(np.float32)
    turb = cv2.GaussianBlur(cv2.resize(turb, (W, H)), (0, 0), 8)
    turb = 0.85 + 0.3 * (turb - turb.mean())
    A = np.array([0.78, 0.62, 0.38], np.float32) * turb[..., None]   # RGB ochre
    out = clean * t + A * (1 - t)
    gray = out.mean(-1, keepdims=True)
    out = out * (1 - 0.25 * severity) + gray * (0.25 * severity)
    return np.clip(out, 0, 1)

def add_snow(clean, severity=0.5, depth01=None, seed=0, max_depth=6.0):
    """Snow: cool bright depth-aware veil + two flake layers + slight downward streak."""
    clean = _as_float_rgb(clean); H, W = clean.shape[:2]
    rng = np.random.default_rng(seed)
    if depth01 is None:
        depth01 = np.tile(np.linspace(0.2, 1.0, H)[:, None], (1, W)).astype(np.float32)
    d = np.clip(depth01, 0, 1) * max_depth
    t = np.clip(np.exp(-(0.2 + severity * 0.7) * d), 0.15, 1.0)[..., None]
    base = clean * t + np.array([0.90, 0.92, 0.96], np.float32) * (1 - t)
    flakes = np.zeros((H, W), np.float32)
    for scale, count, blur in [(3, int(severity * 1200), 1.2), (2, int(severity * 2500), 0.6)]:
        n = max(0, count)
        ys = rng.integers(0, H, n); xs = rng.integers(0, W, n)
        f = np.zeros((H, W), np.float32); f[ys, xs] = rng.uniform(0.6, 1.0, n).astype(np.float32)
        f = cv2.GaussianBlur(cv2.dilate(f, np.ones((scale, scale), np.uint8)), (0, 0), blur)
        flakes = np.maximum(flakes, f)
    k = 7; mk = np.zeros((k, k), np.float32); cv2.line(mk, (k // 2, 0), (k // 2, k - 1), 1.0, 1); mk /= mk.sum()
    flakes = np.maximum(flakes, cv2.filter2D(flakes, -1, mk) * 0.6)
    return np.clip(base + np.stack([flakes] * 3, -1) * 0.9, 0, 1)

# severity-parameterised dispatch (single "severity dial" per weather)
WEATHER_FNS = {
    "fog":   lambda img, s, depth: add_fog(img, depth, beta=0.2 + s * 1.3),
    "rain":  lambda img, s, depth: add_rain(img, intensity=s),
    "night": lambda img, s, depth: add_night(img, severity=s),
    "glare": lambda img, s, depth: add_glare(img, severity=s),
    "dust":  lambda img, s, depth: add_dust(img, severity=s, depth01=depth),
    "snow":  lambda img, s, depth: add_snow(img, severity=s, depth01=depth),
}

def apply_weather(img_rgb01, weather, severity, depth01):
    return WEATHER_FNS[weather](img_rgb01, float(severity), depth01)

print("weather fns:", list(WEATHER_FNS))


## 2 · Depth estimation (Depth‑Anything‑V2, with fallback)

Fog/dust/snow are depth‑aware. We estimate a relative depth map once per source image and cache it.
Depth‑Anything‑V2 outputs *inverse* depth (bright = near), so we invert it to the `0=near,1=far`
convention the weather functions expect. If the model can't load (no internet), we fall back to a
vertical gradient prior (`top = far`).

In [ ]:

class DepthEstimator:
    def __init__(self, model="depth-anything/Depth-Anything-V2-Small-hf"):
        self.pipe = None
        try:
            from transformers import pipeline
            self.pipe = pipeline("depth-estimation", model=model,
                                 device=0 if torch.cuda.is_available() else -1)
            print("Depth-Anything-V2 loaded.")
        except Exception as e:
            print("Depth model unavailable, using gradient fallback:", repr(e)[:120])

    def __call__(self, img_rgb01):
        H, W = img_rgb01.shape[:2]
        if self.pipe is None:
            return np.tile(np.linspace(1.0, 0.0, H)[:, None], (1, W)).astype(np.float32)
        pil = Image.fromarray((img_rgb01 * 255).astype(np.uint8))
        out = self.pipe(pil)
        depth = np.asarray(out["depth"], np.float32)
        depth = cv2.resize(depth, (W, H))
        d = (depth - depth.min()) / (depth.ptp() + 1e-6)   # normalize -> bright=near
        return np.clip(1.0 - d, 0, 1).astype(np.float32)   # invert -> 0=near,1=far

DEPTH = DepthEstimator()

def get_depth_cached(img_path, img_rgb01, cache_dir):
    cache_dir = Path(cache_dir); cache_dir.mkdir(parents=True, exist_ok=True)
    f = cache_dir / (Path(img_path).stem + ".npy")
    if f.exists():
        return np.load(f)
    d = DEPTH(img_rgb01); np.save(f, d); return d


## 3 · Data — source images, class remap, and a leak‑free split

**Pitfall fixed:** we split the *clean* images into train/val/test **first**, then apply weather to
each split independently — never the other way around.

- **Real BDD100K:** set `CFG.bdd_root` to a directory containing `images/` and YOLO `labels/`.
- **Default (demo):** auto‑download **COCO128** and keep only driving‑relevant classes, remapped to our 8.


In [ ]:

def _read_rgb(path):
    bgr = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if bgr is None:
        raise IOError(f"could not read {path}")
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0

def save_rgb(path, arr01):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Image.fromarray((np.clip(arr01, 0, 1) * 255).astype(np.uint8)).save(str(path), quality=90)

# COCO id -> our id (car,person,bicycle,motorcycle,bus,truck,traffic_light,stop_sign)
COCO2OURS = {0: 1, 1: 2, 2: 0, 3: 3, 5: 4, 7: 5, 9: 6, 11: 7}

def load_coco128():
    from ultralytics.data.utils import check_det_dataset
    info = check_det_dataset("coco128.yaml")            # downloads if needed
    base = Path(info["path"])
    img_dir = base / "images" / "train2017"
    lbl_dir = base / "labels" / "train2017"
    items = []
    for lbl in sorted(lbl_dir.glob("*.txt")):
        img = img_dir / (lbl.stem + ".jpg")
        if not img.exists():
            continue
        rows = []
        for line in lbl.read_text().strip().splitlines():
            p = line.split()
            if len(p) < 5:
                continue
            cid = int(float(p[0]))
            if cid in COCO2OURS:
                rows.append([COCO2OURS[cid], *[float(x) for x in p[1:5]]])
        if rows:                                         # keep only images with our classes
            items.append((str(img), rows))
    return items

def load_bdd(root):
    root = Path(root)
    img_dir = next((root / d for d in ["images", "images/train", "train/images"]
                    if (root / d).exists()), None)
    lbl_dir = next((root / d for d in ["labels", "labels/train", "train/labels"]
                    if (root / d).exists()), None)
    assert img_dir and lbl_dir, "BDD root must contain images/ and labels/ (YOLO format)"
    items = []
    for lbl in sorted(Path(lbl_dir).glob("*.txt")):
        img = next((Path(img_dir) / (lbl.stem + e) for e in [".jpg", ".png", ".jpeg"]
                    if (Path(img_dir) / (lbl.stem + e)).exists()), None)
        if img is None:
            continue
        rows = [[int(float(x[0]))] + [float(v) for v in x[1:5]]
                for x in (l.split() for l in lbl.read_text().strip().splitlines()) if len(x) >= 5]
        if rows:
            items.append((str(img), rows))
    return items

# --- load source items ---
if CFG.bdd_root and Path(CFG.bdd_root).exists():
    ITEMS = load_bdd(CFG.bdd_root); print("Loaded BDD:", len(ITEMS))
else:
    ITEMS = load_coco128(); print("Loaded COCO128 (demo):", len(ITEMS))

if CFG.QUICK_DEMO:
    ITEMS = ITEMS[:CFG.n_source_quick]

random.Random(SEED).shuffle(ITEMS)
n = len(ITEMS); n_tr = int(0.7 * n); n_va = int(0.15 * n)
SPLITS = {"train": ITEMS[:n_tr], "val": ITEMS[n_tr:n_tr + n_va], "test": ITEMS[n_tr + n_va:]}
print({k: len(v) for k, v in SPLITS.items()})


In [ ]:

DEPTH_CACHE = CFG.root / "depth_cache"

def write_label(path, rows):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text("\n".join(f"{int(c)} {x:.6f} {y:.6f} {w:.6f} {h:.6f}"
                                    for c, x, y, w, h in rows))

def build_clean_split(split_name, items, out_root):
    """Write clean images+labels for a split; return list of (img_path, rows, depth_path)."""
    idir = out_root / "images" / split_name
    ldir = out_root / "labels" / split_name
    idir.mkdir(parents=True, exist_ok=True); ldir.mkdir(parents=True, exist_ok=True)
    recs = []
    for img_path, rows in items:
        rgb = _read_rgb(img_path)
        depth = get_depth_cached(img_path, rgb, DEPTH_CACHE)
        stem = Path(img_path).stem
        save_rgb(idir / f"{stem}.jpg", rgb)
        write_label(ldir / f"{stem}.txt", rows)
        recs.append((img_path, rows, rgb.shape, depth))
    return recs

print("depth+clean writing helpers ready")


In [ ]:

def write_yaml(path, root, names, train="images/train", val="images/val"):
    lines = [f"path: {Path(root).resolve()}", f"train: {train}", f"val: {val}", "names:"]
    lines += [f"  {i}: {nm}" for i, nm in enumerate(names)]
    Path(path).write_text("\n".join(lines)); return str(path)

def gen_weather_copies(items, split_name, out_root, weathers, severities,
                       clean_frac=1.0, tag=""):
    """Apply each (weather,severity) to each item, writing images+labels into a YOLO tree.
       clean_frac: also emit this fraction of clean copies (prevents forgetting)."""
    idir = out_root / "images" / split_name
    ldir = out_root / "labels" / split_name
    idir.mkdir(parents=True, exist_ok=True); ldir.mkdir(parents=True, exist_ok=True)
    count = 0
    for img_path, rows in items:
        rgb = _read_rgb(img_path)
        depth = get_depth_cached(img_path, rgb, DEPTH_CACHE)
        stem = Path(img_path).stem
        if random.random() < clean_frac:
            save_rgb(idir / f"{stem}{tag}_clean.jpg", rgb)
            write_label(ldir / f"{stem}{tag}_clean.txt", rows)
            count += 1
        for w in weathers:
            for s in severities:
                out = apply_weather(rgb, w, s, depth)
                key = f"{stem}{tag}_{w}_{int(s*100):03d}"
                save_rgb(idir / f"{key}.jpg", out)
                write_label(ldir / f"{key}.txt", rows)
                count += 1
    return count

print("synthetic writer ready")


## 4 · YOLOv8 training wrapper

`yolov8n` by default (fast, so we can afford many cycles); the capacity controller can grow it to
`s`/`m` when accuracy plateaus. One helper trains from a given starting checkpoint and returns the
`best.pt` path.

In [ ]:

from ultralytics import YOLO

def train_yolo(data_yaml, start_ckpt, name, epochs):
    model = YOLO(start_ckpt)
    model.train(data=data_yaml, epochs=epochs, imgsz=CFG.imgsz, batch=CFG.batch,
                device=DEVICE, seed=SEED, project=str(CFG.root / "runs"), name=name,
                exist_ok=True, verbose=False, plots=False, pretrained=True)
    return model, str(CFG.root / "runs" / name / "weights" / "best.pt")


## 5 · Evaluation, severity sweep & failure map

**Bug fixed:** the original `results.per_class_ap[...]` isn't the real Ultralytics API. We read
per‑class AP@0.5 from `results.box.ap50` aligned with `results.box.ap_class_index`.
For each `(weather, severity)` we build a small validation set on the **test** split and record
per‑class AP → a failure map that flags weak cells.

In [ ]:

def per_class_ap(results, names):
    """Return {class_name: AP@0.5 (%)} robustly across Ultralytics versions."""
    out = {nm: float("nan") for nm in names}
    box = getattr(results, "box", None)
    if box is None:
        return out
    try:
        idx = list(np.array(box.ap_class_index).astype(int))
        ap50 = np.array(box.ap50).astype(float)          # aligned with idx
        for j, ci in enumerate(idx):
            if 0 <= ci < len(names):
                out[names[ci]] = float(ap50[j]) * 100.0
    except Exception:
        pass
    return out

def make_sweep_set(weather, severity, out_root):
    """Create a val set = test-split images corrupted by one (weather,severity)."""
    sub = out_root / f"{weather}_{int(severity*100):03d}"
    idir = sub / "images" / "val"; ldir = sub / "labels" / "val"
    if idir.exists(): shutil.rmtree(sub)
    idir.mkdir(parents=True, exist_ok=True); ldir.mkdir(parents=True, exist_ok=True)
    for img_path, rows in SPLITS["test"]:
        rgb = _read_rgb(img_path)
        depth = get_depth_cached(img_path, rgb, DEPTH_CACHE)
        stem = Path(img_path).stem
        save_rgb(idir / f"{stem}.jpg", apply_weather(rgb, weather, severity, depth))
        write_label(ldir / f"{stem}.txt", rows)
    return write_yaml(sub / "data.yaml", sub, CLASS_NAMES, train="images/val", val="images/val")

def severity_sweep(model, out_root, weathers, severities):
    """Return failure_map[(weather,severity)] = {class: ap%}."""
    out_root = Path(out_root); out_root.mkdir(parents=True, exist_ok=True)
    fmap = {}
    for w in weathers:
        for s in severities:
            yaml = make_sweep_set(w, s, out_root)
            res = model.val(data=yaml, imgsz=CFG.imgsz, device=DEVICE,
                            verbose=False, plots=False, save_json=False)
            fmap[(w, s)] = per_class_ap(res, CLASS_NAMES)
    return fmap

def mean_ap(fmap):
    vals = [v for cell in fmap.values() for v in cell.values() if not math.isnan(v)]
    return float(np.mean(vals)) if vals else 0.0

def weak_cells(fmap, thr):
    weak = []
    for (w, s), cell in fmap.items():
        for cls, ap in cell.items():
            if not math.isnan(ap) and ap < thr:
                weak.append((w, s, cls, ap))
    return sorted(weak, key=lambda x: x[3])


In [ ]:

import matplotlib.pyplot as plt

def plot_failure_map(fmap, weathers, severities, title="Failure map (AP@0.5 %)"):
    fig, axes = plt.subplots(1, len(weathers), figsize=(3.2 * len(weathers), 3.6), squeeze=False)
    for a, w in zip(axes[0], weathers):
        grid = np.array([[fmap[(w, s)].get(c, np.nan) for s in severities] for c in CLASS_NAMES])
        im = a.imshow(grid, vmin=0, vmax=70, cmap="RdYlGn", aspect="auto")
        a.set_title(w); a.set_xticks(range(len(severities)))
        a.set_xticklabels([f"{s:.1f}" for s in severities], fontsize=7)
        a.set_yticks(range(len(CLASS_NAMES))); a.set_yticklabels(CLASS_NAMES, fontsize=7)
        for i in range(grid.shape[0]):
            for j in range(grid.shape[1]):
                if not np.isnan(grid[i, j]):
                    a.text(j, i, f"{grid[i,j]:.0f}", ha="center", va="center", fontsize=6)
    fig.suptitle(title); fig.colorbar(im, ax=axes[0].tolist(), shrink=0.7, label="AP%")
    plt.tight_layout(); plt.show()


## 6 · Self‑adaptation loop (targeted oversampling + grow / prune)

Each cycle: read the failure map → oversample weak `(weather, severity)` cells (factor grows with how
weak they are, capped) → rebuild a mixed training set (**30% clean · 20% easy · 50% hard‑targeted** to
avoid catastrophic forgetting) → fine‑tune → re‑evaluate.

**Capacity controller (the "add blocks or prune" part):**
- If mean‑AP gain **plateaus** and the model is still small → **grow** capacity (`n → s → m`), transferring
  learning by starting the next fine‑tune from the larger pretrained backbone. (Growing a YOLO graph
  in place would break skip‑connections, so we grow by variant escalation — the safe, faithful analogue
  of "adding blocks/parameters".)
- After the final cycle → **prune** redundant weights (next section) and fine‑tune to recover.


In [ ]:

def oversample_factor(ap, thr, cap):
    if math.isnan(ap) or ap >= thr:
        return 1
    return int(min(cap, 1 + math.ceil((thr - ap) / thr * 4)))

def build_targeted_trainset(fmap, out_root, cycle):
    """Mixed set: clean + easy-uniform + oversampled weak cells (on the TRAIN split)."""
    root = out_root / f"adapt_cycle_{cycle}"
    if root.exists(): shutil.rmtree(root)
    (root / "images/train").mkdir(parents=True, exist_ok=True)
    (root / "labels/train").mkdir(parents=True, exist_ok=True)
    # val = clean val split (kept fixed so mAP is comparable across cycles)
    build_clean_split("val", SPLITS["val"], root)

    tr = SPLITS["train"]
    # 30% clean
    gen_weather_copies(tr, "train", root, weathers=[], severities=[], clean_frac=1.0, tag="_cl")
    # 20% "easy": low severity, all weathers, half the images
    easy_items = tr[: max(1, len(tr) // 2)]
    gen_weather_copies(easy_items, "train", root, weathers=list(CFG.weathers),
                       severities=[min(CFG.train_severities)], clean_frac=0.0, tag="_ez")
    # 50% "hard-targeted": oversample weak cells
    weak = weak_cells(fmap, CFG.weak_ap_threshold)
    # aggregate weak (weather,severity) -> max factor across its weak classes
    cell_factor = {}
    for w, s, cls, ap in weak:
        f = oversample_factor(ap, CFG.weak_ap_threshold, CFG.max_oversample)
        cell_factor[(w, s)] = max(cell_factor.get((w, s), 1), f)
    for (w, s), fac in cell_factor.items():
        for rep in range(fac):
            gen_weather_copies(tr, "train", root, weathers=[w], severities=[s],
                               clean_frac=0.0, tag=f"_t{rep}")
    yaml = write_yaml(root / "data.yaml", root, CLASS_NAMES)
    return yaml, cell_factor


In [ ]:

GROW_CHAIN = {"yolov8n.pt": "yolov8s.pt", "yolov8s.pt": "yolov8m.pt", "yolov8m.pt": "yolov8m.pt"}

def self_adaptation(baseline_ckpt, baseline_fmap):
    history = [{"cycle": 0, "mean_ap": mean_ap(baseline_fmap),
                "variant": CFG.yolo_variant, "action": "baseline", "fmap": baseline_fmap}]
    ckpt = baseline_ckpt
    variant = CFG.yolo_variant
    fmap = baseline_fmap
    for cycle in range(1, CFG.n_cycles + 1):
        yaml, cell_factor = build_targeted_trainset(fmap, CFG.root / "adapt", cycle)
        print(f"[cycle {cycle}] oversampled weak cells:",
              {f"{w}@{s}": f for (w, s), f in cell_factor.items()})

        prev = history[-1]["mean_ap"]
        grew = False
        # plateau? grow capacity by starting fine-tune from the larger pretrained backbone
        if cycle >= 2 and (history[-1]["mean_ap"] - history[-2]["mean_ap"]) < CFG.plateau_delta \
           and GROW_CHAIN[variant] != variant:
            variant = GROW_CHAIN[variant]; start = variant; grew = True
            print(f"[cycle {cycle}] plateau -> GROW capacity to {variant}")
        else:
            start = ckpt

        model, ckpt = train_yolo(yaml, start, name=f"adapt_c{cycle}", epochs=CFG.epochs_adapt)
        fmap = severity_sweep(model, CFG.root / "sweep" / f"c{cycle}",
                              CFG.weathers, CFG.sweep_severities)
        m = mean_ap(fmap)
        history.append({"cycle": cycle, "mean_ap": m, "variant": variant,
                        "action": "grow" if grew else "targeted", "fmap": fmap,
                        "ckpt": ckpt})
        print(f"[cycle {cycle}] mean AP {prev:.1f} -> {m:.1f}  (variant={variant})")
    return history, ckpt, model


## 7 · Self‑improvement by pruning (weights **and** outputs)

Two complementary strategies, both validated to keep tensor shapes intact (safe with YOLO skip‑connections):
1. **Magnitude / weight‑based** — L1‑unstructured prune of every `Conv2d` (`prune.l1_unstructured` → `remove` to bake in the zeros).
2. **Output / activation‑based** — forward hooks record mean `|activation|` per output channel over a
   calibration batch; the lowest‑scoring channels are zeroed (structured mask).
A short fine‑tune recovers accuracy; we report sparsity vs. mAP so you can pick an operating point.

In [ ]:

import torch.nn as nn
import torch.nn.utils.prune as prune

def conv_sparsity(torch_model):
    z = t = 0
    for m in torch_model.modules():
        if isinstance(m, nn.Conv2d):
            w = m.weight.detach(); z += int((w == 0).sum()); t += w.numel()
    return 100.0 * z / max(1, t)

def magnitude_prune(model, amount):
    tm = model.model
    for m in tm.modules():
        if isinstance(m, nn.Conv2d) and m.weight.requires_grad:
            prune.l1_unstructured(m, name="weight", amount=amount)
            prune.remove(m, "weight")          # bake zeros in permanently
    return model

def activation_prune(model, data_yaml, amount=0.15, n_batches=3):
    """Zero the lowest-activation output channels of each Conv2d (output-based)."""
    tm = model.model
    convs = [m for m in tm.modules() if isinstance(m, nn.Conv2d)]
    scores = {i: None for i in range(len(convs))}
    def mk(i):
        def hook(mod, inp, out):
            s = out.detach().abs().mean(dim=(0, 2, 3))
            scores[i] = s if scores[i] is None else scores[i] + s
        return hook
    handles = [c.register_forward_hook(mk(i)) for i, c in enumerate(convs)]
    # calibration forward passes on a few sweep images
    from ultralytics.data.utils import check_det_dataset
    info = check_det_dataset(data_yaml)
    val_dir = Path(info["path"]) / info["val"]
    imgs = list(Path(val_dir).glob("*.jpg"))[: n_batches * CFG.batch]
    tm.eval()
    with torch.no_grad():
        for p in imgs:
            model.predict(str(p), imgsz=CFG.imgsz, device=DEVICE, verbose=False)
    for h in handles: h.remove()
    with torch.no_grad():
        for i, c in enumerate(convs):
            if scores[i] is None or c.out_channels < 8:
                continue
            k = int(amount * c.out_channels)
            if k <= 0:
                continue
            idx = torch.argsort(scores[i])[:k]
            c.weight[idx] = 0.0
            if c.bias is not None: c.bias[idx] = 0.0
    return model

def prune_and_recover(ckpt, data_yaml, amount, name="pruned"):
    model = YOLO(ckpt)
    before = conv_sparsity(model.model)
    magnitude_prune(model, amount)
    activation_prune(model, data_yaml, amount=0.10)
    after = conv_sparsity(model.model)
    # fine-tune to recover
    model.train(data=data_yaml, epochs=max(3, CFG.epochs_adapt // 2), imgsz=CFG.imgsz,
                batch=CFG.batch, device=DEVICE, seed=SEED,
                project=str(CFG.root / "runs"), name=name, exist_ok=True,
                verbose=False, plots=False)
    print(f"Conv sparsity {before:.1f}% -> {after:.1f}%")
    return model, str(CFG.root / "runs" / name / "weights" / "best.pt")


## 8 · (Optional) Test‑time adaptation — BatchNorm recalibration

For unseen conditions, briefly put BatchNorm layers in train mode and forward a few test images so
the running mean/var re‑estimate to the new distribution, then predict.

In [ ]:

def tta_bn_adapt(model, image_paths, steps=3, momentum=0.1):
    tm = model.model
    for m in tm.modules():
        if isinstance(m, nn.BatchNorm2d):
            m.train(); m.momentum = momentum
    with torch.no_grad():
        for _ in range(steps):
            for p in image_paths:
                model.predict(str(p), imgsz=CFG.imgsz, device=DEVICE, verbose=False)
    for m in tm.modules():
        if isinstance(m, nn.BatchNorm2d):
            m.eval()
    return model


## 9 · Real‑world validation (RTTS / DAWN / ExDark)

Attach a real adverse‑weather set as a Kaggle dataset in **YOLO format** and point `val_data` at its
yaml. This is the honest test of whether synthetic training transferred.

In [ ]:

def validate_real(ckpt, val_yaml):
    if not Path(val_yaml).exists():
        print("No real-world yaml at", val_yaml, "— attach RTTS/ExDark/DAWN and set the path.")
        return None
    model = YOLO(ckpt)
    res = model.val(data=val_yaml, imgsz=CFG.imgsz, device=DEVICE, verbose=False, plots=False)
    print("Real-world mAP@0.5:", round(float(res.box.map50) * 100, 2),
          "| per-class:", {k: round(v, 1) for k, v in per_class_ap(res, CLASS_NAMES).items()})
    return res


## 10 · Run the whole pipeline

In [ ]:

t0 = time.time()

# 1) baseline synthetic training set (train split, all weathers over the train grid)
BASE = CFG.root / "baseline"
if BASE.exists(): shutil.rmtree(BASE)
build_clean_split("val", SPLITS["val"], BASE)
print("baseline train imgs:",
      gen_weather_copies(SPLITS["train"], "train", BASE, list(CFG.weathers),
                         list(CFG.train_severities), clean_frac=1.0))
base_yaml = write_yaml(BASE / "data.yaml", BASE, CLASS_NAMES)

# 2) train baseline
base_model, base_ckpt = train_yolo(base_yaml, CFG.yolo_variant, "baseline",
                                   CFG.epochs_baseline)

# 3) baseline failure map
base_fmap = severity_sweep(base_model, CFG.root / "sweep" / "c0",
                           CFG.weathers, CFG.sweep_severities)
print("baseline mean AP:", round(mean_ap(base_fmap), 2))
plot_failure_map(base_fmap, CFG.weathers, CFG.sweep_severities, "Cycle 0 (baseline)")

# 4) self-adaptation (targeted oversampling + capacity growth)
history, final_ckpt, final_model = self_adaptation(base_ckpt, base_fmap)
plot_failure_map(history[-1]["fmap"], CFG.weathers, CFG.sweep_severities,
                 f"Final (cycle {history[-1]['cycle']}, {history[-1]['variant']})")

# 5) prune + recover
pruned_model, pruned_ckpt = prune_and_recover(
    final_ckpt, base_yaml, CFG.prune_amount, "pruned")
pruned_fmap = severity_sweep(pruned_model, CFG.root / "sweep" / "pruned",
                             CFG.weathers, CFG.sweep_severities)
print("pruned mean AP:", round(mean_ap(pruned_fmap), 2))

print(f"\nTotal wall time: {(time.time()-t0)/60:.1f} min")


In [ ]:

cycles = [h["cycle"] for h in history]
means = [h["mean_ap"] for h in history]
plt.figure(figsize=(6, 4))
plt.plot(cycles, means, "-o")
for h in history:
    plt.annotate(h["action"], (h["cycle"], h["mean_ap"]),
                 textcoords="offset points", xytext=(0, 8), fontsize=8, ha="center")
plt.xlabel("adaptation cycle"); plt.ylabel("mean AP@0.5 (%) over weather×severity")
plt.title("Self-improvement curve"); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

print("Summary")
for h in history:
    print(f"  cycle {h['cycle']}: meanAP={h['mean_ap']:.1f}  variant={h['variant']}  ({h['action']})")
print(f"  pruned : meanAP={mean_ap(pruned_fmap):.1f}  sparsity={conv_sparsity(pruned_model.model):.1f}%")


In [ ]:

# Point this at an attached real adverse-weather dataset (YOLO format) to test transfer.
REAL_YAML = "/kaggle/input/your-rtts-or-exdark/data.yaml"
validate_real(final_ckpt, REAL_YAML)


## 11 · Scaling up & notes

- **Use real BDD100K:** attach it, set `CFG.bdd_root`, set `CFG.QUICK_DEMO=False`. Severity grids,
  epochs and cycles automatically widen.
- **Speed on P100:** `yolov8n` + `imgsz=512` keeps cycles short. Raise `imgsz` to 640 and use `yolov8s`
  for final numbers.
- **Bugs fixed from the doc's pseudocode:** night `np.power` NaN + channel order; rain zero‑sum kernel;
  fog depth convention/scale; real Ultralytics per‑class AP API; TTA/pruning use `model.model`.
- **Faithful vs. simplified:** physics weather, depth‑aware fog/dust/snow, failure maps, targeted
  oversampling, magnitude + activation pruning are faithful. "Adding blocks" is done as safe **variant
  growth** (n→s→m) rather than in‑place graph surgery, which would break YOLO's skip‑connections.
- **Avoid label leakage:** the split happens on *clean* images before any weather is applied.
